# Predicting NBA Success Based on Rookie Stats

This notebook represents my workflow for getting data from database in usable format for modeling. There will be another notebook for modeling workflow and feature selection.

Author: Rasheed Martin

In [121]:
import sys
sys.path.append('/Users/rashe/PycharmProjects/PredictingNBACareers')
from utils.duckdb import get_duckdb_connection
import duckdb
import polars as pl

## Let's first get the data for players

### Get NBA Players First Three Season. (Context, we only have data from 2000-2025 so we may have players who started before that but we are treating their first years in the 2000s as their first year)

I will also be able to get their third year stats. So we will do rookie stats, aggregated first three years and third year stats.

In [ ]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_season_counts AS (
    SELECT
        player_id,
        COUNT(DISTINCT season_id) AS season_count
    FROM player_stat_values
    GROUP BY player_id
    
),
first_three_seasons AS (
    SELECT
        psv.player_id,
        s.label,
        ROW_NUMBER() OVER (
            PARTITION BY psv.player_id
            ORDER BY s.label
        ) AS season_rank
    FROM player_stat_values psv
    JOIN player_season_counts psc
      ON psv.player_id = psc.player_id
     JOIN seasons s on psv.season_id = s.season_id
    GROUP BY psv.player_id, s.label
),
selected_stats AS (
    SELECT
        psv.*
    FROM player_stat_values psv
    JOIN seasons s on psv.season_id = s.season_id
    JOIN first_three_seasons fts
      ON psv.player_id = fts.player_id
     AND s.label = fts.label
    WHERE fts.season_rank <= 3
)
SELECT 
    p.name AS Player,
    s.label AS Season,
    t.abbreviation AS Team,
    st.name AS Stat_Type,
    ps.name AS Stat_Name,
    selected_stats.value AS Stat_Value
FROM selected_stats
JOIN players p ON selected_stats.player_id = p.player_id
JOIN seasons s ON selected_stats.season_id = s.season_id
JOIN teams t ON selected_stats.team_id = t.team_id
JOIN stat_types st ON selected_stats.stat_type_id = st.stat_type_id
JOIN stats ps ON selected_stats.stat_id = ps.stat_id
WHERE st.stat_type_id in (2, 3)
ORDER BY p.name, s.label, stat_name;
        """
    ).arrow()
    df_first_three_years = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [123]:
df_first_three_years

Player,Season,Team,Stat_Type,Stat_Name,Stat_Value
str,i32,str,str,str,f32
"""AC. Green""",1985,"""LAL""","""Total_Player_Data""","""2P""",208.0
"""AC. Green""",1985,"""LAL""","""Total_Player_Data""","""2P%""",0.545
"""AC. Green""",1985,"""LAL""","""Total_Player_Data""","""2PA""",382.0
"""AC. Green""",1985,"""LAL""","""Total_Player_Data""","""3P""",1.0
"""AC. Green""",1985,"""LAL""","""Total_Player_Data""","""3P%""",0.167
…,…,…,…,…,…
"""Zyon Pullin""",2024,"""MEM""","""Advanced_Data""","""USG%""",13.4
"""Zyon Pullin""",2024,"""MEM""","""Advanced_Data""","""VORP""",0.0
"""Zyon Pullin""",2024,"""MEM""","""Advanced_Data""","""WS""",0.0


In [124]:
df_first_three_years = df_first_three_years.pivot(
    index=["Player", "Season"],
    on="Stat_Name",
    values="Stat_Value",
    aggregate_function="first"
)
df_first_three_years

Player,Season,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,Player_Age,STL,STL%,TOV,TOV%,TRB,TRB%,TS%,Trp-Dbl,USG%,VORP,WS,WS/48,eFG%
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",1985,208.0,0.545,382.0,1.0,0.167,6.0,0.015,54.0,4.2,49.0,1.7,-1.6,0.1,221.0,15.5,2.0,209.0,0.539,388.0,102.0,0.611,167.0,0.43,82.0,1.0,1542.0,-1.6,160.0,12.4,1.4,11.8,229.0,521.0,22.0,49.0,1.5,99.0,17.700001,381.0,14.0,0.564,0.0,14.7,0.2,3.3,0.103,0.54
"""AC. Green""",1986,316.0,0.543,582.0,0.0,0.0,5.0,0.009,84.0,4.6,80.0,2.0,1.0,0.1,405.0,18.799999,3.3,316.0,0.538,587.0,220.0,0.78,282.0,0.48,79.0,72.0,2240.0,0.9,210.0,11.2,4.3,15.7,171.0,852.0,23.0,70.0,1.5,102.0,12.5,615.0,15.3,0.599,0.0,14.7,1.7,7.6,0.163,0.538
"""AC. Green""",1987,322.0,0.505,638.0,0.0,0.0,2.0,0.003,93.0,4.5,45.0,1.0,0.3,-0.1,465.0,19.1,3.4,322.0,0.503,640.0,293.0,0.773,379.0,0.592,82.0,64.0,2636.0,0.4,245.0,11.1,4.5,14.5,204.0,937.0,24.0,87.0,1.6,120.0,12.9,710.0,15.3,0.581,0.0,14.7,1.5,7.9,0.144,0.503
"""AJ Griffin""",2022,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,19.0,42.0,1.4,42.0,7.1,153.0,6.0,0.577,0.0,17.9,0.4,2.2,0.075,0.56
"""AJ Griffin""",2023,8.0,0.348,23.0,10.0,0.256,39.0,0.629,5.0,3.7,2.0,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zydrunas Ilgauskas""",1998,29.0,0.509,57.0,0.0,null,0.0,0.0,4.0,4.9,7.0,3.3,0.0,-0.8,27.0,20.0,0.2,29.0,0.509,57.0,18.0,0.6,30.0,0.526,5.0,5.0,171.0,0.8,17.0,12.8,0.3,17.5,24.0,76.0,23.0,4.0,1.3,9.0,11.4,44.0,16.4,0.541,0.0,22.6,0.1,0.5,0.134,0.509
"""Zydrunas Ilgauskas""",2000,114.0,0.491,232.0,0.0,0.0,2.0,0.009,18.0,5.4,37.0,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616.0,-1.4,65.0,12.3,0.1,16.0,78.0,281.0,25.0,15.0,1.3,60.0,18.299999,160.0,15.0,0.524,0.0,24.1,0.0,0.7,0.056,0.487
"""Zylan Cheatham""",2019,6.0,0.75,8.0,0.0,0.0,1.0,0.111,3.0,7.7,1.0,1.6,-7.2,-0.5,6.0,12.5,0.0,6.0,0.667,9.0,0.0,null,0.0,0.0,4.0,0.0,51.0,-6.6,3.0,6.2,0.0,6.7,10.0,12.0,24.0,1.0,0.9,4.0,30.799999,9.0,9.4,0.667,0.0,10.4,-0.1,0.0,0.034,0.667


Going to filter out rookies and other players who did not play 3 or more years

In [125]:
# players_who_played_first_three_years = df_first_three_years.select("Player").unique().to_series().to_list()

Below we calculate the percent growth from year 1 to year 2 and year 2 to year 3 and then the avg annual year over year growth. The resulting data frame will have their rookie stats included

### Getting general players roy info, all rookie team info, draft pick, draft position

In [126]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
SELECT 
    p.name AS Player,
    p.draft_pick AS Draft_Pick,
    p.draft_class AS Draft_Class,
    r.rank AS ROY_Rank,
    r.first_place_votes AS ROY_Votes,
    r.point_share AS ROY_Point_Share,
    a.rank AS All_Rookie_Rank,
    a.first_place_votes AS All_Rookie_First_Place_Votes,
    a.second_place_votes AS All_Rookie_Second_Place_Votes,
    a.points_share AS All_Rookie_Point_Share

FROM nba_data.main.players p
LEFT JOIN nba_data.main.roy_voting r on p.player_id = r.player_id 
LEFT JOIN nba_data.main.all_rookie_voting a on p.player_id = a.player_id
        """
    ).arrow()
    df_players_rookie_awards = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [127]:
df_players_rookie_awards

Player,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share
str,i32,i32,i32,i32,f32,str,i32,i32,f32
"""Kevin Durant""",2,2007,1,90,0.872,"""1ST Team""",28,1,0.983
"""Elfrid Payton""",10,2014,4,3,0.122,"""1ST Team""",121,8,0.962
"""Brook Lopez""",10,2008,3,2,0.212,"""1ST Team""",21,7,0.845
"""Greg Monroe""",7,2010,6,0,0.025,"""2ND Team""",13,16,0.724
"""Gorgui Dieng""",21,2013,6,0,0.005,"""2ND Team""",21,54,0.384
…,…,…,…,…,…,…,…,…,…
"""Jorge Garbajosa""",null,null,8,0,0.027,"""1ST Team""",13,11,0.638
"""Juan Carlos Navarro""",40,2002,6,0,0.005,"""2ND Team""",5,14,0.414
"""OJ Mayo""",null,null,2,5,0.41,"""1ST Team""",29,0,1.0


Let's merge their rookie awards with the pct differential and rookie stats

In [128]:
import polars as pl

# List your stat columns
stat_cols = df_first_three_years.select(pl.exclude(["Player", "Season", "Player_Age"])).columns

# Build list of aggregate expressions
exprs = []

for stat in stat_cols:
    col = pl.col(stat)
    exprs.extend([
        # Year‑1→2 growth
        ((col.shift(-1).over("Player") - col) / col.shift(-1).over("Player") * 100)
            .alias(f"{stat}_growth_y1_y2"),
        # Year‑2→3 growth
        ((col.shift(-2).over("Player") - col.shift(-1).over("Player"))
         / col.shift(-2).over("Player") * 100)
            .alias(f"{stat}_growth_y2_y3"),
        # Average of the two
        ((( (col.shift(-1).over("Player") - col) / col.shift(-1).over("Player") * 100 )
          + ( (col.shift(-2).over("Player") - col.shift(-1).over("Player")) / col.shift(-2).over("Player") * 100 )
         ) / 2).alias(f"{stat}_avg_yoy_growth")
    ])

# Apply to only first-year rows per player
df_pct_diff_three_years = (
    df_first_three_years.select(["Player", "Season", "Player_Age"] + stat_cols)
    .sort(["Player", "Season"])
    .fill_null(0)
    .with_columns(exprs)
    .filter(pl.col("Season") == pl.col("Season").min().over("Player"))  # keep only the year‑1 row
)

df_pct_diff_three_years

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,STL_avg_yoy_growth,STL%_growth_y1_y2,STL%_growth_y2_y3,STL%_avg_yoy_growth,TOV_growth_y1_y2,TOV_growth_y2_y3,TOV_avg_yoy_growth,TOV%_growth_y1_y2,TOV%_growth_y2_y3,TOV%_avg_yoy_growth,TRB_growth_y1_y2,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",1985,22.0,208.0,0.545,382.0,1.0,0.167,6.0,0.015,54.0,4.2,49.0,1.7,-1.6,0.1,221.0,15.5,2.0,209.0,0.539,388.0,102.0,0.611,167.0,0.43,82.0,1.0,1542.0,-1.6,160.0,12.4,1.4,11.8,229.0,521.0,49.0,…,24.770115,0.0,6.250001,3.125001,2.941176,15.000001,8.970589,-41.600006,3.100772,-19.249617,38.048779,13.380281,25.714531,8.496733,0.0,4.248366,5.843066,-3.098108,1.372479,NaN,NaN,NaN,0.0,0.0,0.0,88.235291,-13.333337,37.450977,56.578953,3.797471,30.188211,36.809818,-13.19445,11.807684,-0.371754,-6.958243,-3.664999
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,null,-366.666626,null,null,-425.0,null,null,37.16814,null,null,-750.0,null,null,-5.263161,null,null,-51.047127,null,null,NaN,null,null,-3.468211,null,null,233.333344,null,null,650.000061,null,null,162.5,null,null,-50.943398,null,null
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""AJ. Bramlett""",1999,23.0,4.0,0.19,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,0.0,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,1.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""AJ. English""",1990,23.0,248.0,0.458,541.0,3.0,0.097,31.0,0.054,177.0,18.0,15.0,0.6,-3.9,-1.9,81.0,6.2,0.7,251.0,0.439,572.0,111.0,0.707,157.0,0.274,70.0,12.0,1443.0,-2.1,66.0,5.0,-0.5,10.6,127.0,616.0,25.0,…,null,0.0,null,null,-28.089888,null,null,-71.590912,null,null,12.5,null,null,-1.81818,null,null,0.0,null,null,NaN,null,null,14.843755,null,null,12.500003,null,null,77.777779,null,null,68.0,null,null,-1.146795,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5

In [129]:
df_pct_diff_rookie_stats = df_pct_diff_three_years.join(
    df_players_rookie_awards,
    on="Player",
    how="left"
)
df_pct_diff_rookie_stats

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,TOV%_avg_yoy_growth,TRB_growth_y1_y2,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,str,i32,i32,f32
"""AC. Green""",1985,22.0,208.0,0.545,382.0,1.0,0.167,6.0,0.015,54.0,4.2,49.0,1.7,-1.6,0.1,221.0,15.5,2.0,209.0,0.539,388.0,102.0,0.611,167.0,0.43,82.0,1.0,1542.0,-1.6,160.0,12.4,1.4,11.8,229.0,521.0,49.0,…,-19.249617,38.048779,13.380281,25.714531,8.496733,0.0,4.248366,5.843066,-3.098108,1.372479,NaN,NaN,NaN,0.0,0.0,0.0,88.235291,-13.333337,37.450977,56.578953,3.797471,30.188211,36.809818,-13.19445,11.807684,-0.371754,-6.958243,-3.664999,null,null,null,null,null,null,null,null,null
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,null,-750.0,null,null,-5.263161,null,null,-51.047127,null,null,NaN,null,null,-3.468211,null,null,233.333344,null,null,650.000061,null,null,162.5,null,null,-50.943398,null,null,16,2022,null,null,null,"""Received Votes""",0,26,0.13
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,23,2024,null,null,null,null,null,null,null
"""AJ. Bramlett""",1999,23.0,4.0,0.19,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,0.0,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,1.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""AJ. English""",1990,23.0,248.0,0.458,541.0,3.0,0.097,31.0,0.054,177.0,18.0,15.0,0.6,-3.9,-1.9,81.0,6.2,0.7,251.0,0.439,572.0,111.0,0.707,157.0,0.274,70.0,12.0,1443.0,-2.1,66.0,5.0,-0.5,10.6,127.0,616.0,25.0,…,null,12.5,null,null,-1.81818,null,null,0.0,null,null,NaN,null,null,14.843755,null,null,12.500003,null,null,77.777779,null,null,68.0,null,null,-1.146795,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5,0.1,10.0,70.0,153.0,13.0,…,-5.656522,20.289854,6.756756,13.

Do some feature creation

### Getting team data to see if they made the playoffs prior

In [130]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_seasons AS (
    SELECT DISTINCT
        psv.player_id,
        psv.team_id,
        psv.season_id
    FROM player_stat_values psv
)

SELECT 
    p.name AS player_name,
    s.label AS season,
    t.name AS team,
    CASE 
        WHEN tsi_prev.made_playoffs IS NULL THEN NULL
        ELSE tsi_prev.made_playoffs
    END AS team_made_playoffs_last_year
FROM player_seasons ps
JOIN players p ON p.player_id = ps.player_id
JOIN teams t ON t.team_id = ps.team_id
JOIN seasons s ON s.season_id = ps.season_id
JOIN seasons s_prev ON s_prev.label = s.label - 1
LEFT JOIN team_season_info tsi_prev 
    ON tsi_prev.team_id = ps.team_id 
    AND tsi_prev.season_id = s_prev.season_id
        """
    ).arrow()
    df_team_playoffs_prior = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [131]:
df_team_playoffs_prior = df_team_playoffs_prior.filter(pl.col("season") == pl.col("season").min().over("player_name")) # only rookies' first season

# Step 1: Count unique teams per player per season
team_counts = (
    df_team_playoffs_prior.group_by(["player_name", "season"])
      .agg(pl.col("team").n_unique().alias("team_count"))
)

# Step 2: Join this count back to the original DataFrame
df_team_playoffs_prior = df_team_playoffs_prior.join(team_counts, on=["player_name", "season"])

# Step 3: Create a new column with the simplified team info
df_team_playoffs_prior = df_team_playoffs_prior.with_columns([
    pl.when(pl.col("team_count") > 1)
      .then(pl.lit("Multiple Teams"))
      .otherwise(pl.col("team"))
      .alias("team_simplified")
]).drop("team_count")


df_team_playoffs_prior = (
    df_team_playoffs_prior.group_by(["player_name", "season"])
      .agg([
          pl.col("team_simplified").first(),  # or use .max() or .unique() depending on context
          *[
              pl.col(col).first()
              for col in df_team_playoffs_prior.columns
              if col not in ["player_name", "season", "team", "team_simplified"]
          ]
      ])
)

df_team_playoffs_prior = df_team_playoffs_prior.rename({
    "team_simplified": "Team",
    "player_name": "Player",
    "season": "Season"
})

df_team_playoffs_prior


Player,Season,Team,team_made_playoffs_last_year
str,i32,str,bool
"""Clifford Ray""",1977,"""Golden State Warriors""",null
"""Isaiah Rider""",1993,"""Minnesota Timberwolves""",null
"""Reggie Jackson""",2011,"""Oklahoma City Thunder""",true
"""Blake Wesley""",2022,"""San Antonio Spurs""",false
"""Sam Hauser""",2021,"""Boston Celtics""",true
…,…,…,…
"""Dee Brown""",1990,"""Boston Celtics""",null
"""Lamar Odom""",1999,"""Los Angeles Clippers""",null
"""Johnny Rogers""",1986,"""Sacramento Kings""",null


In [132]:
df_with_team_player_stats = df_pct_diff_rookie_stats.join(df_team_playoffs_prior, on=["Player", "Season"], how="left")
df_with_team_player_stats

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,TRB_growth_y2_y3,TRB_avg_yoy_growth,TRB%_growth_y1_y2,TRB%_growth_y2_y3,TRB%_avg_yoy_growth,TS%_growth_y1_y2,TS%_growth_y2_y3,TS%_avg_yoy_growth,Trp-Dbl_growth_y1_y2,Trp-Dbl_growth_y2_y3,Trp-Dbl_avg_yoy_growth,USG%_growth_y1_y2,USG%_growth_y2_y3,USG%_avg_yoy_growth,VORP_growth_y1_y2,VORP_growth_y2_y3,VORP_avg_yoy_growth,WS_growth_y1_y2,WS_growth_y2_y3,WS_avg_yoy_growth,WS/48_growth_y1_y2,WS/48_growth_y2_y3,WS/48_avg_yoy_growth,eFG%_growth_y1_y2,eFG%_growth_y2_y3,eFG%_avg_yoy_growth,Draft_Pick,Draft_Class,ROY_Rank,ROY_Votes,ROY_Point_Share,All_Rookie_Rank,All_Rookie_First_Place_Votes,All_Rookie_Second_Place_Votes,All_Rookie_Point_Share,Team,team_made_playoffs_last_year
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,f32,str,i32,i32,f32,str,bool
"""AC. Green""",1985,22.0,208.0,0.545,382.0,1.0,0.167,6.0,0.015,54.0,4.2,49.0,1.7,-1.6,0.1,221.0,15.5,2.0,209.0,0.539,388.0,102.0,0.611,167.0,0.43,82.0,1.0,1542.0,-1.6,160.0,12.4,1.4,11.8,229.0,521.0,49.0,…,13.380281,25.714531,8.496733,0.0,4.248366,5.843066,-3.098108,1.372479,NaN,NaN,NaN,0.0,0.0,0.0,88.235291,-13.333337,37.450977,56.578953,3.797471,30.188211,36.809818,-13.19445,11.807684,-0.371754,-6.958243,-3.664999,null,null,null,null,null,null,null,null,null,"""Los Angeles Lakers""",null
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,null,null,-5.263161,null,null,-51.047127,null,null,NaN,null,null,-3.468211,null,null,233.333344,null,null,650.000061,null,null,162.5,null,null,-50.943398,null,null,16,2022,null,null,null,"""Received Votes""",0,26,0.13,"""Atlanta Hawks""",true
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,23,2024,null,null,null,null,null,null,null,"""Multiple Teams""",true
"""AJ. Bramlett""",1999,23.0,4.0,0.19,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,0.0,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,1.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Cleveland Cavaliers""",null
"""AJ. English""",1990,23.0,248.0,0.458,541.0,3.0,0.097,31.0,0.054,177.0,18.0,15.0,0.6,-3.9,-1.9,81.0,6.2,0.7,251.0,0.439,572.0,111.0,0.707,157.0,0.274,70.0,12.0,1443.0,-2.1,66.0,5.0,-0.5,10.6,127.0,616.0,25.0,…,null,null,-1.81818,null,null,0.0,null,null,NaN,null,null,14.843755,null,null,12.500003,null,null,77.777779,null,null,68.0,null,null,-1.146795,null,null,null,null,null,null,null,null,null,null,null,"""Unknown""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""Unknown""",null
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.

### Get third year stats

In [133]:
df_three_year_stats = df_first_three_years.join(
    df_first_three_years.group_by("Player").agg(pl.col("Season").max().alias("Season")),
    on=["Player", "Season"]
)
df_three_year_stats

Player,Season,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,Player_Age,STL,STL%,TOV,TOV%,TRB,TRB%,TS%,Trp-Dbl,USG%,VORP,WS,WS/48,eFG%
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",1987,322.0,0.505,638.0,0.0,0.0,2.0,0.003,93.0,4.5,45.0,1.0,0.3,-0.1,465.0,19.1,3.4,322.0,0.503,640.0,293.0,0.773,379.0,0.592,82.0,64.0,2636.0,0.4,245.0,11.1,4.5,14.5,204.0,937.0,24.0,87.0,1.6,120.0,12.9,710.0,15.3,0.581,0.0,14.7,1.5,7.9,0.144,0.503
"""AJ Griffin""",2023,8.0,0.348,23.0,10.0,0.256,39.0,0.629,5.0,3.7,2.0,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Bramlett""",1999,4.0,0.19,21.0,0.0,null,0.0,0.0,0.0,0.0,0.0,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,null,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,23.0,1.0,0.8,3.0,12.5,22.0,20.1,0.19,0.0,17.1,-0.2,-0.2,-0.129,0.19
"""AJ. English""",1991,360.0,0.443,812.0,6.0,0.176,34.0,0.04,143.0,13.7,9.0,0.3,-3.8,-2.2,94.0,6.3,0.7,366.0,0.433,846.0,148.0,0.841,176.0,0.208,81.0,6.0,1665.0,-1.6,74.0,4.8,0.1,12.5,160.0,886.0,24.0,32.0,0.9,89.0,8.8,168.0,5.5,0.48,0.0,25.6,-0.8,0.9,0.025,0.436
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,25.0,2.0,1.4,5.0,13.4,8.0,6.2,0.435,0.0,22.9,-0.1,-0.1,-0.042,0.417
"""Zoran Planinic""",2005,53.0,0.417,127.0,13.0,0.232,56.0,0.306,53.0,15.0,4.0,0.5,-5.7,0.4,60.0,12.0,0.8,66.0,0.361,183.0,46.0,0.697,66.0,0.361,56.0,1.0,596.0,-6.1,14.0,2.7,-0.8,6.9,91.0,191.0,23.0,20.0,1.8,43.0,16.9,74.0,7.3,0.45,0.0,20.200001,-0.6,0.1,0.005,0.396
"""Zydrunas Ilgauskas""",2000,114.0,0.491,232.0,0.0,0.0,2.0,0.009,18.0,5.4,37.0,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616.0,-1.4,65.0,12.3,0.1,16.0,78.0,281.0,25.0,15.0,1.3,60.0,18.299999,160.0,15.0,0.524,0.0,24.1,0.0,0.7,0.056,0.487


In [134]:
df_without_target = df_with_team_player_stats.join(df_three_year_stats.rename({col: f"{col}_third" for col in df_first_three_years.columns if col not in  ["Player"]}), on=["Player"], how="left")
df_without_target

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,BLK%_third,BPM_third,DBPM_third,DRB_third,DRB%_third,DWS_third,FG_third,FG%_third,FGA_third,FT_third,FT%_third,FTA_third,FTr_third,G_third,GS_third,MP_third,OBPM_third,ORB_third,ORB%_third,OWS_third,PER_third,PF_third,PTS_third,Player_Age_third,STL_third,STL%_third,TOV_third,TOV%_third,TRB_third,TRB%_third,TS%_third,Trp-Dbl_third,USG%_third,VORP_third,WS_third,WS/48_third,eFG%_third
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",1985,22.0,208.0,0.545,382.0,1.0,0.167,6.0,0.015,54.0,4.2,49.0,1.7,-1.6,0.1,221.0,15.5,2.0,209.0,0.539,388.0,102.0,0.611,167.0,0.43,82.0,1.0,1542.0,-1.6,160.0,12.4,1.4,11.8,229.0,521.0,49.0,…,1.0,0.3,-0.1,465.0,19.1,3.4,322.0,0.503,640.0,293.0,0.773,379.0,0.592,82.0,64.0,2636.0,0.4,245.0,11.1,4.5,14.5,204.0,937.0,24.0,87.0,1.6,120.0,12.9,710.0,15.3,0.581,0.0,14.7,1.5,7.9,0.144,0.503
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Bramlett""",1999,23.0,4.0,0.19,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,0.0,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,1.0,…,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,null,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,23.0,1.0,0.8,3.0,12.5,22.0,20.1,0.19,0.0,17.1,-0.2,-0.2,-0.129,0.19
"""AJ. English""",1990,23.0,248.0,0.458,541.0,3.0,0.097,31.0,0.054,177.0,18.0,15.0,0.6,-3.9,-1.9,81.0,6.2,0.7,251.0,0.439,572.0,111.0,0.707,157.0,0.274,70.0,12.0,1443.0,-2.1,66.0,5.0,-0.5,10.6,127.0,616.0,25.0,…,0.3,-3.8,-2.2,94.0,6.3,0.7,366.0,0.433,846.0,148.0,0.841,176.0,0.208,81.0,6.0,1665.0,-1.6,74.0,4.8,0.1,12.5,160.0,886.0,24.0,32.0,0.9,89.0,8.8,168.0,5.5,0.48,0.0,25.6,-0.8,0.9,0.025,0.436
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,25.0,2.0,1.4,5.0,13.4,8.0,6.2,0.435,0.0,22.9,-0.1,-0.1,-0.042,0.417
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5,0.1,10.0,70.0,153.0,13.0,…,0.5,-5.7,0.4,60.0,12.0,0.8,66.0,0.361,183.0,46.0,0.697,66.0,0.361,56.0,1.0,596.0,-6.1,14.0,2.7,-0.8,6.9,91.0,191.0,23.0,20.0,1.8,43.0,16.9,74.0,7.3,0.45,0.0,20.200001,-0.6,0.1,0.005,0.396
"""Zydrunas Ilgauskas""",1997,22.0,453.0,0.519,872.0,1.0,0.25,4.0,0.005,71.0,5.7,135.0,4.3,1.4,0.5,444.0,22.6,4.9,454.0,0.518,876.0,230.0,0.762,302.0,0.345,82.0,81.0,2379.0,1.0,279.0,14.6,4.5,19.700001,288.0,1139.0,52.0,…,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616

Looking at the null count, it is obvious based on context that if the players have null values at any step they did not take any or have any. 

From the above analysis, we can see that the following pairs of features have the highest absolute correlation:
Field Goals and Points
2 Points Attempted vs 2 Points Made
Minutes Per Games

In [135]:
with get_duckdb_connection() as conn:
    arr = conn.execute(
        """
WITH player_stats as (
SELECT
    MAX(CASE WHEN st.name = 'WS' THEN psv.value ELSE NULL END) AS ws,
    MAX(CASE WHEN st.name = 'USG%' THEN psv.value ELSE NULL END) AS usage,
	MAX(CASE WHEN st.name = 'G' THEN psv.value ELSE NULL END) AS games_played,
	MAX(CASE WHEN st.name = 'MP' THEN psv.value ELSE NULL END) AS minutes_played,
	MAX(CASE WHEN st.name = 'GS' THEN psv.value ELSE NULL END) AS games_started,
	MAX(CASE WHEN st.name = 'BPM' THEN psv.value ELSE NULL END) AS box_plus_minus,
	psv.season_id,
	psv.player_id,
	psv.team_id
FROM
	nba_data.main.player_stat_values psv
JOIN nba_data.main.stats st on
	st.stat_id = psv.stat_id
JOIN nba_data.main.stat_types s on s.stat_type_id = psv.stat_type_id
WHERE psv.stat_type_id in (3, 2) -- Only Total and Advanced Stats
GROUP BY psv.player_id, psv.season_id, psv.team_id
)
SELECT
 p.name as Player,
 ps.ws as Win_Share,
 ps."usage" as Usage,
 ps.games_played as Games_Played,
 ps.minutes_played as Minutes_Played,
 ps.games_started as Games_Started,
 ps.box_plus_minus as Box_Plus_Minus,
 a.all_nba_team as All_Nba_Team,
 a.allstar as AllStar,
 a.six_moy_rank as Six_MOY,
 s."label" as Season,
 t.name As Team
 
FROM nba_data.main.award_info a
JOIN nba_data.main.players p on a.player_id = p.player_id
JOIN player_stats ps on ps.player_id = p.player_id AND ps.season_id = a.season_id
JOIN nba_data.main.seasons s on s.season_id = ps.season_id
JOIN nba_data.main.teams t on t.team_id = a.team_id
        """
    ).arrow()
    df_award_info = pl.from_arrow(arr)

Connected to: /Users/rashe/PycharmProjects/PredictingNBACareers/data/refined_raw_data/nba_data.duckdb


In [136]:
df_award_info

Player,Win_Share,Usage,Games_Played,Minutes_Played,Games_Started,Box_Plus_Minus,All_Nba_Team,AllStar,Six_MOY,Season,Team
str,f32,f32,f32,f32,f32,f32,i32,bool,i32,i32,str
"""Clifford Ray""",-0.3,12.0,66.0,838.0,null,-5.6,0,false,0,1980,"""Golden State Warriors"""
"""Bob Elliott""",1.6,18.799999,73.0,1320.0,null,-1.9,0,false,0,1980,"""New Jersey Nets"""
"""Bernard King""",9.1,23.200001,81.0,2914.0,null,2.8,0,false,0,1980,"""Golden State Warriors"""
"""Jeff Judkins""",0.7,17.4,62.0,666.0,null,-3.1,0,false,0,1980,"""Utah Jazz"""
"""Andrew Toney""",2.2,25.700001,75.0,1768.0,57.0,-3.0,0,false,0,1980,"""Philadelphia 76ers"""
…,…,…,…,…,…,…,…,…,…,…,…
"""Max Christie""",2.7,15.2,78.0,2127.0,36.0,-2.5,0,false,0,2024,"""Unknown"""
"""Lester Quinones""",-0.1,22.6,13.0,183.0,0.0,-6.1,0,false,0,2024,"""Unknown"""
"""Luka Doncic""",5.9,33.900002,50.0,1769.0,50.0,6.7,0,false,0,2024,"""Dallas Mavericks"""


In [137]:
df_award_info.null_count()

Player,Win_Share,Usage,Games_Played,Minutes_Played,Games_Started,Box_Plus_Minus,All_Nba_Team,AllStar,Six_MOY,Season,Team
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,298,0,0,1183,3,0,0,0,0,0


In [138]:
def min_max_norm(col: str) -> pl.Expr:
    return ((pl.col(col) - pl.col(col).min()) / (pl.col(col).max() - pl.col(col).min())).alias(f"{col}_norm")

Creating the target variable to model against

In [139]:
# Aggregate stats per player
player_summary = df_award_info.group_by("Player").agg([
    pl.col("All_Nba_Team").max().alias("All_Nba_Team_Max"),
    pl.col("AllStar").max().alias("AllStar_Max"),
    pl.col("Games_Played").mean().alias("Games_Played_Mean"),
    pl.col("Win_Share").sum().alias("Win_Share_Sum"),
    pl.col("Box_Plus_Minus").mean().alias("Box_Plus_Minus_Mean"),
    pl.col("Minutes_Played").mean().alias("Minutes_Played_Mean"),
    pl.col("Games_Started").mean().alias("Games_Started_Mean"),
    pl.col("Usage").mean().alias("Usage_Mean")
])
player_summary = player_summary.with_columns([
    (
        0.5 * min_max_norm("Win_Share_Sum") +
        0.4 * min_max_norm("Box_Plus_Minus_Mean") +
        0.4 * min_max_norm("Usage_Mean") +
        0.2 * min_max_norm("Games_Started_Mean") +
        0.1 * min_max_norm("Minutes_Played_Mean")).alias("Player_Score")
])
# Assign player type
five_tier_label = player_summary.with_columns([
    pl.when(pl.col("All_Nba_Team_Max") > 0)
      .then(pl.lit("All NBA Caliber"))
      .when(pl.col("AllStar_Max") > 0)
      .then(pl.lit("All Star Caliber"))
      .when(
          pl.col("Player_Score") > 0.4
      ).then(pl.lit("Star Player"))
      .when(
          pl.col("Player_Score") > 0.3
      ).then(pl.lit("Role Player"))
      .otherwise(pl.lit("Bust Player"))
      .alias("Player_Type")
])

five_tier_label

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Tobias Harris""",0,false,69.285713,75.299995,0.728571,2194.785645,60.714287,21.464285,0.6733,"""Star Player"""
"""Joe Kopicki""",0,false,39.333332,3.5,-0.466667,395.666656,1.0,15.433333,0.297837,"""Bust Player"""
"""Juan Toscano-Anderson""",0,false,43.714287,6.7,-3.485714,646.571411,7.857143,11.942857,0.3015,"""Role Player"""
"""Dontell Jefferson""",0,false,6.0,0.2,-0.1,84.0,0.0,18.0,0.293758,"""Bust Player"""
"""Ken Bannister""",0,false,50.599998,0.6,-4.8,773.400024,14.2,19.120001,0.346212,"""Role Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Dexter Pittman""",0,false,12.5,0.4,-16.75,86.75,1.5,22.799999,0.278417,"""Bust Player"""
"""Dennis Schroder""",0,false,71.133331,42.400002,-1.906667,1955.93335,38.066666,23.68,0.554749,"""Star Player"""
"""Leon Smith""",0,false,7.5,0.2,-1.5,52.0,0.0,19.799999,0.298717,"""Bust Player"""


In [140]:
from pathlib import Path
path = Path.cwd().parent.parent / "data/clean_data" / "player_label.csv"
five_tier_label.select(["Player", "Player_Type"]).write_csv(path)

In [141]:
player_5tier_label_only_three_years = five_tier_label.join(pl.DataFrame({"Player": players_who_played_first_three_years}), on="Player", how="inner")

In [142]:
path = Path.cwd().parent.parent / "data/clean_data" / "player_5tier_label.csv"
five_tier_label.select(["Player", "Player_Type"]).write_csv(path)

Combine this player_label with our full dataset

In [143]:
path = Path.cwd().parent.parent / "data/refined_raw_data" / "Player_Type_Modeling"
path.mkdir(parents=True, exist_ok=True)
df_without_target.join(
    five_tier_label.rename({"Player_Type": "Player_Label"}),
    on="Player",
    how="left"
).write_csv(
    path / "player_5tier_label_with_target.csv"
)

## Updated due to Experimentation with 5 label classification

In [144]:
# Assign player type
three_tier_label = player_summary.with_columns([
    pl.when(pl.col("Player_Score") > 0.4)
      .then(pl.lit("Star Player"))
      .when(
          pl.col("Player_Score") > 0.3
      ).then(pl.lit("Role Player"))
      .otherwise(pl.lit("Bust Player"))
      .alias("Player_Type")
])
three_tier_label

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Tobias Harris""",0,false,69.285713,75.299995,0.728571,2194.785645,60.714287,21.464285,0.6733,"""Star Player"""
"""Joe Kopicki""",0,false,39.333332,3.5,-0.466667,395.666656,1.0,15.433333,0.297837,"""Bust Player"""
"""Juan Toscano-Anderson""",0,false,43.714287,6.7,-3.485714,646.571411,7.857143,11.942857,0.3015,"""Role Player"""
"""Dontell Jefferson""",0,false,6.0,0.2,-0.1,84.0,0.0,18.0,0.293758,"""Bust Player"""
"""Ken Bannister""",0,false,50.599998,0.6,-4.8,773.400024,14.2,19.120001,0.346212,"""Role Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Dexter Pittman""",0,false,12.5,0.4,-16.75,86.75,1.5,22.799999,0.278417,"""Bust Player"""
"""Dennis Schroder""",0,false,71.133331,42.400002,-1.906667,1955.93335,38.066666,23.68,0.554749,"""Star Player"""
"""Leon Smith""",0,false,7.5,0.2,-1.5,52.0,0.0,19.799999,0.298717,"""Bust Player"""


In [145]:
from pathlib import Path
path = Path.cwd().parent.parent / "data/clean_data" / "3_player_label.csv"
three_tier_label.select(["Player", "Player_Type"]).write_csv(path)

In [146]:
player_3tier_label_only_three_years = three_tier_label.join(pl.DataFrame({"Player": players_who_played_first_three_years}), on="Player", how="inner")

In [147]:
player_3tier_label_only_three_years

Player,All_Nba_Team_Max,AllStar_Max,Games_Played_Mean,Win_Share_Sum,Box_Plus_Minus_Mean,Minutes_Played_Mean,Games_Started_Mean,Usage_Mean,Player_Score,Player_Type
str,i32,bool,f32,f32,f32,f32,f32,f32,f32,str
"""Tobias Harris""",0,false,69.285713,75.299995,0.728571,2194.785645,60.714287,21.464285,0.6733,"""Star Player"""
"""Juan Toscano-Anderson""",0,false,43.714287,6.7,-3.485714,646.571411,7.857143,11.942857,0.3015,"""Role Player"""
"""Bo Outlaw""",0,false,60.933334,50.900002,0.8,1382.666626,23.466667,12.28,0.459353,"""Star Player"""
"""Morris Peterson""",0,false,64.63636,35.700001,-1.490909,1760.54541,44.363636,17.854546,0.520128,"""Star Player"""
"""Robert Pack""",0,false,42.46154,14.299999,-1.376923,881.692322,12.076923,22.253847,0.396502,"""Role Player"""
…,…,…,…,…,…,…,…,…,…,…
"""Trenton Hassell""",0,false,71.555557,13.799999,-3.244444,1808.666626,47.555557,12.1,0.453058,"""Star Player"""
"""Trevelin Queen""",0,false,15.5,0.8,-2.25,184.75,0.5,19.549999,0.302181,"""Role Player"""
"""Dexter Pittman""",0,false,12.5,0.4,-16.75,86.75,1.5,22.799999,0.278417,"""Bust Player"""


In [148]:
path = Path.cwd().parent.parent / "data/refined_raw_data" / "Player_Type_Modeling"
path.mkdir(parents=True, exist_ok=True)
df_without_target.join(
    three_tier_label.rename({"Player_Type": "Player_Label"}),
    on="Player",
    how="left"
).write_csv(
    path / "player_3tier_label_with_target_all_years.csv"
)

In [149]:
df_without_target

Player,Season,Player_Age,2P,2P%,2PA,3P,3P%,3PA,3PAr,AST,AST%,BLK,BLK%,BPM,DBPM,DRB,DRB%,DWS,FG,FG%,FGA,FT,FT%,FTA,FTr,G,GS,MP,OBPM,ORB,ORB%,OWS,PER,PF,PTS,STL,…,BLK%_third,BPM_third,DBPM_third,DRB_third,DRB%_third,DWS_third,FG_third,FG%_third,FGA_third,FT_third,FT%_third,FTA_third,FTr_third,G_third,GS_third,MP_third,OBPM_third,ORB_third,ORB%_third,OWS_third,PER_third,PF_third,PTS_third,Player_Age_third,STL_third,STL%_third,TOV_third,TOV%_third,TRB_third,TRB%_third,TS%_third,Trp-Dbl_third,USG%_third,VORP_third,WS_third,WS/48_third,eFG%_third
str,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""AC. Green""",1985,22.0,208.0,0.545,382.0,1.0,0.167,6.0,0.015,54.0,4.2,49.0,1.7,-1.6,0.1,221.0,15.5,2.0,209.0,0.539,388.0,102.0,0.611,167.0,0.43,82.0,1.0,1542.0,-1.6,160.0,12.4,1.4,11.8,229.0,521.0,49.0,…,1.0,0.3,-0.1,465.0,19.1,3.4,322.0,0.503,640.0,293.0,0.773,379.0,0.592,82.0,64.0,2636.0,0.4,245.0,11.1,4.5,14.5,204.0,937.0,24.0,87.0,1.6,120.0,12.9,710.0,15.3,0.581,0.0,14.7,1.5,7.9,0.144,0.503
"""AJ Griffin""",2022,19.0,147.0,0.536,274.0,101.0,0.39,259.0,0.486,73.0,7.0,12.0,0.7,-0.8,-0.5,116.0,9.2,0.9,248.0,0.465,533.0,42.0,0.894,47.0,0.088,72.0,12.0,1401.0,-0.3,37.0,2.9,1.3,12.6,87.0,639.0,42.0,…,1.1,-9.6,-3.5,16.0,10.6,0.0,18.0,0.29,62.0,2.0,1.0,2.0,0.032,20.0,0.0,171.0,-6.0,2.0,1.2,-0.4,1.2,6.0,48.0,20.0,1.0,0.3,8.0,11.3,18.0,5.7,0.382,0.0,17.299999,-0.3,-0.4,-0.12,0.371
"""AJ Johnson""",2024,20.0,58.0,0.472,123.0,24.0,0.267,90.0,0.423,76.0,17.1,3.0,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,12.0,…,0.4,-8.1,-2.5,51.0,8.3,0.0,82.0,0.385,213.0,32.0,0.865,37.0,0.174,29.0,11.0,639.0,-5.6,8.0,1.3,-0.5,7.1,50.0,220.0,20.0,12.0,0.9,35.0,13.2,59.0,4.8,0.48,0.0,17.4,-1.0,-0.4,-0.032,0.441
"""AJ. Bramlett""",1999,23.0,4.0,0.19,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,0.0,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,1.0,…,0.0,-15.9,-6.7,10.0,18.5,0.1,4.0,0.19,21.0,0.0,null,0.0,0.0,8.0,0.0,61.0,-9.2,12.0,21.700001,-0.2,-0.4,13.0,8.0,23.0,1.0,0.8,3.0,12.5,22.0,20.1,0.19,0.0,17.1,-0.2,-0.2,-0.129,0.19
"""AJ. English""",1990,23.0,248.0,0.458,541.0,3.0,0.097,31.0,0.054,177.0,18.0,15.0,0.6,-3.9,-1.9,81.0,6.2,0.7,251.0,0.439,572.0,111.0,0.707,157.0,0.274,70.0,12.0,1443.0,-2.1,66.0,5.0,-0.5,10.6,127.0,616.0,25.0,…,0.3,-3.8,-2.2,94.0,6.3,0.7,366.0,0.433,846.0,148.0,0.841,176.0,0.208,81.0,6.0,1665.0,-1.6,74.0,4.8,0.1,12.5,160.0,886.0,24.0,32.0,0.9,89.0,8.8,168.0,5.5,0.48,0.0,25.6,-0.8,0.9,0.025,0.436
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Zoran Dragic""",2014,25.0,8.0,0.5,16.0,3.0,0.214,14.0,0.467,5.0,11.2,0.0,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,2.0,…,0.0,-7.7,-3.6,3.0,4.6,0.0,11.0,0.367,30.0,3.0,0.6,5.0,0.167,16.0,1.0,75.0,-4.1,5.0,7.8,-0.1,8.2,6.0,28.0,25.0,2.0,1.4,5.0,13.4,8.0,6.2,0.435,0.0,22.9,-0.1,-0.1,-0.042,0.417
"""Zoran Planinic""",2003,21.0,44.0,0.454,97.0,9.0,0.281,32.0,0.248,68.0,23.9,3.0,0.5,-3.4,-0.1,41.0,10.2,0.6,53.0,0.411,129.0,38.0,0.633,60.0,0.465,49.0,1.0,473.0,-3.3,14.0,3.5,0.1,10.0,70.0,153.0,13.0,…,0.5,-5.7,0.4,60.0,12.0,0.8,66.0,0.361,183.0,46.0,0.697,66.0,0.361,56.0,1.0,596.0,-6.1,14.0,2.7,-0.8,6.9,91.0,191.0,23.0,20.0,1.8,43.0,16.9,74.0,7.3,0.45,0.0,20.200001,-0.6,0.1,0.005,0.396
"""Zydrunas Ilgauskas""",1997,22.0,453.0,0.519,872.0,1.0,0.25,4.0,0.005,71.0,5.7,135.0,4.3,1.4,0.5,444.0,22.6,4.9,454.0,0.518,876.0,230.0,0.762,302.0,0.345,82.0,81.0,2379.0,1.0,279.0,14.6,4.5,19.700001,288.0,1139.0,52.0,…,4.4,-1.8,-0.4,95.0,17.799999,0.7,114.0,0.487,234.0,53.0,0.679,78.0,0.333,24.0,24.0,616